In [1]:
from ppopt.mplp_program import MPLP_Program
from ppopt.mpmodel import MPModeler
from ppopt.mp_solvers.solve_mpqp import solve_mpqp, mpqp_algorithm
# import itertools as itools
import time
import numpy as np
from collections import defaultdict
from scipy.optimize import linprog
import chaospy as cp
from typing import List, Callable, Union
from numpy.polynomial.legendre import leggauss

In [2]:
def theta_interval_at_point(solution, theta_vector: np.ndarray, max_idx: int = 0, min_idx: int = 1) -> tuple:
    """Given the parametric solution for theta_k and the current 'state' vector (theta_prev + d),
        return the scalar lower and upper bound [t_min, t_max] for this theta_k.

    Args:
        solution (_type_): _description_
        t_vector (np.ndarray): _description_
        max_idx (int, optional): _description_. Defaults to 0.
        min_idx (int, optional): _description_. Defaults to 1.

    Returns:
        tuple: _description_
    """

    theta_vector_aug = np.append(theta_vector, 1).reshape(-1, 1)

    if isinstance(solution, list):
        theta_min = solution[0]
        theta_max = solution[1]
        return float(theta_min), float(theta_max)

    for region in solution.critical_regions:
        if region.is_inside(theta_vector.reshape(-1, 1)):
            coefficients = np.concatenate([region.A, region.b], axis=1)[:2, :]
            max_coefficients = coefficients[max_idx]
            min_coefficients = coefficients[min_idx]
            theta_max = (max_coefficients @ theta_vector_aug).item()
            theta_min = (min_coefficients @ theta_vector_aug).item()
            return theta_min, theta_max

    raise ValueError(
        "The provided theta_vector is not inside any critical region of the solution.")


def map_u_to_theta_and_jacobian(solutions: List, u: np.ndarray, d_vector: np.ndarray = None) -> tuple:
    """Given the parametric solutions for theta_k and the current 'state' vector (theta_prev + d),
        return the theta_k vector and the Jacobian matrix dtheta/du.
    Args:
        solutions (List): List of parametric solutions for each theta_k.
        u (np.ndarray): 1D arraay of canonical coordinates 
        d_vector (np.ndarray): Current disturbance vector.
    """
    theta_values = []
    jacobian = 1.0

    for k, sol in enumerate(solutions):
        if isinstance(d_vector, np.ndarray):
            theta_vector = np.block([np.array(theta_values), d_vector])
        else:
            theta_vector = np.array(theta_values, dtype=float)

        theta_min, theta_max = theta_interval_at_point(sol, theta_vector)
        length = theta_max - theta_min
        
        theta_k = 0.5 * length * u[k] + 0.5 * (theta_max + theta_min)
        theta_values.append(theta_k)

        jacobian *= 0.5 * length

    return np.array(theta_values, dtype=float), jacobian


def calculate_stocflexibility_smolyak(solutions: List, level: int, joint_func: Callable[[List[float]], float], d_vector: np.ndarray = None, rule: str = "gaussian") -> float:
    """Compute stochastic flexibility using a Smolyak sparse grid in canonical  u-space

    Args:
        solutions (List): list of solutions for each theta dimension (same structure as in calculate_stocflexibility)
        level (int): Smolyak level (1,2,3,...) controls accuracy & number of points
        joint_func (Callable[[List[float]], float]): callable f(theta_list) -> scalar
        d_vector (np.ndarray, optional):design vector (np.ndarray). Defaults to None.
        rule (str, optional): 1D quadrature rule passed to chaospy (e.g. "gaussian"). Defaults to "gaussian".

    Returns:
        float: _description_
    """

    n_theta = len(solutions)

    dist = cp.J(*[cp.Uniform(-1, 1) for _ in range(n_theta)])

    nodes_u, weights_expectation = cp.quadrature.sparse_grid(
        order=level, dist=dist, rule=rule)

    weights_u = weights_expectation * (2.0 ** n_theta)

    nodes_u = nodes_u.T

    start = time.time()
    stochastic_flexibility = 0.0

    for i in range(nodes_u.shape[0]):
        u_vector = nodes_u[i, :]
        theta_vector, jacobian = map_u_to_theta_and_jacobian(
            solutions, u_vector, d_vector)
        func_value = joint_func(theta_vector)
 
        stochastic_flexibility += func_value * jacobian * weights_u[i]

    end = time.time()
    print(
        f"Smolyak stochastic flexibility computed in {end - start:.4f} seconds.")
    return stochastic_flexibility

In [3]:
def gauss_legendre_between_bounds(expr_coeffs: np.ndarray, n_gl: int, max_idx: int = 0, min_idx: int = 1):
    """
    Generate n Gauss–Legendre quadrature points and weights between min and max bounds
    defined by two linear expressions.

    Parameters:
        expr_coeffs (np.ndarray): 2xD array. Row 0 = max point co`efficients, Row 1 = min.
        n (int): Number of quadrature points.

    Returns:
        points (np.ndarray): (n, D) array of quadrature points.
        weights (np.ndarray): (n,) array of weights.
    """
    if expr_coeffs.shape[0] != 2:
        raise ValueError("expr_coeffs must have two rows")

    max_coeffs = expr_coeffs[max_idx]
    min_coeffs = expr_coeffs[min_idx]

    # Get Gauss–Legendre points and weights on [-1, 1]
    nodes, weights = leggauss(n_gl)
    weights = weights.reshape(-1, 1)

    # Affine transformation to domain [min_coeffs, max_coeffs]
    points = 0.5 * (np.outer((nodes + 1), max_coeffs) +
                    np.outer((1 - nodes), min_coeffs))

    # Adjust weights to match new domain
    weights = 0.5 * weights@(max_coeffs - min_coeffs).reshape(1, -1)

    return points, weights


def get_quadrature_points(solution, nq: int, t_vector: np.ndarray):
    # Augment t_vector once
    t_vector_aug = np.append(t_vector, 1).reshape(-1, 1)

    if isinstance(solution, list):
        qpoints, qweights = np.polynomial.legendre.leggauss(nq)
        min, max = solution[0], solution[1]
        qps_mapped = 0.5*(max*(1+qpoints) + min*(1-qpoints))
        qws_mapped = 0.5*(max-min)*qweights
        # print(max, min, qps_mapped, qws_mapped)
        return max, min, qps_mapped, qws_mapped

    for region in solution.critical_regions:
        if region.is_inside(t_vector.reshape(-1, 1)):
            coeffs = np.concatenate([region.A, region.b], axis=1)[:2, :]
            qpoints, qweights = gauss_legendre_between_bounds(
                expr_coeffs=coeffs, n_gl=nq)
            return coeffs[0] @ t_vector_aug, coeffs[1] @ t_vector_aug, qpoints @ t_vector_aug, qweights @ t_vector_aug

    # print(f't_vector: {t_vector}')
    # print(f'solution:{solution}')
    raise ValueError("No region found that contains the given t_vector.")


def calculate_stocflexibility(
    sols,
    nq: Union[int, list],
    joint_func,
    d_vector: np.ndarray = None,
    verbose: bool = True,
):
    # Validate nq if it's a list
    if isinstance(nq, list):
        if len(nq) != len(sols):
            raise ValueError(
                "If nq is a list, it must have the same length as sols")

    start_time = time.perf_counter()

    def recurse(level: int, theta_prev: list, weight_prev: float) -> float:
        if level == len(sols):
            return weight_prev * joint_func(theta_prev)

        nql = nq[level] if isinstance(nq, list) else nq

        t_vector = (
            np.block([np.array(theta_prev), d_vector])
            if isinstance(d_vector, np.ndarray)
            else np.array(theta_prev)
        )

        _, _, t_points, t_weights = get_quadrature_points(
            solution=sols[level],
            nq=nql,
            t_vector=t_vector,
        )

        t_points = t_points.flatten()
        t_weights = t_weights.flatten()

        return sum(
            recurse(level + 1, theta_prev + [v], weight_prev * w)
            for v, w in zip(t_points, t_weights)
        )

    stflex = recurse(level=0, theta_prev=[], weight_prev=1.0)

    end_time = time.perf_counter()

    if verbose:
        print(f"Gaussian Legendre Stochastic Flexibility Elapsed time: {end_time - start_time:.4f} s"
              )

    return stflex

In [4]:
def mpformulate_theta_bounds(flex_sol, num_theta: int, theta_bounds: list, num_design: int = 0, design_bounds: list = None, psi_idx: int = 0, theta_m: int = 0):
    A0, b0, F0 = np.empty((len(flex_sol), num_theta)), np.empty(
        (len(flex_sol), 1)), np.empty((len(flex_sol), num_design))
    num_cr = len(flex_sol.critical_regions)
    for i, region in enumerate(flex_sol.critical_regions):
        A0[i] = region.A[psi_idx, :num_theta]
        b0[i] = -region.b[psi_idx]
        F0[i] = -region.A[psi_idx, num_theta:num_theta+num_design]
    # print(f'num_cr:{num_cr}')
    # print(f'num_theta:{num_theta}')
    # print(f'num_design:{num_design}')
    # print(f"A0: {A0}")
    # print(f"b0: {b0}")
    # print(f"F0: {F0}")

    c = np.hstack([np.array([-1, 1]).reshape(1, -1),
                  np.zeros((1, 2 * (num_theta - 1 - theta_m)))]).reshape(-1, 1)
    # print(f'c:{c}')
    # print(f'c.shape: {c.shape}')

    row1_block = np.hstack([block for i in range(theta_m, num_theta)
                           for block in (A0[:, [i]], np.zeros((num_cr, 1)))])
    row2_block = np.hstack([block for i in range(theta_m, num_theta)
                           for block in (np.zeros((num_cr, 1)), A0[:, [i]])])
    bound_row = np.hstack([np.array([-1, 1]).reshape(1, -1),
                          np.zeros((1, 2 * (num_theta - 1 - theta_m)))])
    A = np.vstack([row1_block, row2_block, bound_row, -
                  np.eye(2*(num_theta-theta_m)), np.eye(2*(num_theta-theta_m))])
    # print(f'A: {A}')
    # print(f'A.shape: {A.shape}')

    x_lb = np.array([val for i in range(theta_m, len(theta_bounds))
                    for val in [theta_bounds[i][0]] * 2])
    x_ub = np.array([val for i in range(theta_m, len(theta_bounds))
                    for val in [theta_bounds[i][1]] * 2])
    b = np.vstack([b0, b0, np.zeros((1, 1)), -
                  x_lb.reshape(-1, 1), x_ub.reshape(-1, 1)])
    # print(f'b: {b}')
    # print(f'b.shape: {b.shape}')

    if F0.size == 0 and theta_m == 0:
        # print('here')
        return A, b, c, np.array([]), np.array([]), np.array([]), np.array([])

    F = np.vstack([F0, F0, np.zeros((1, num_design)), np.zeros(
        (4*(num_theta-theta_m), num_design))]) if num_design > 0 else np.vstack([F0, F0])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')
    if theta_m > 0:
        F_lltheta = np.hstack([A0[:, [i]] for i in range(theta_m)])
        # print(f'F_lltheta: {F_lltheta}')
        # print(f'F_lltheta.shape: {F_lltheta.shape}')
        F = np.hstack([np.vstack([-F_lltheta, -F_lltheta, np.zeros((1, len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))]), F]
                      ) if F.size > 0 else np.vstack([-F_lltheta, -F_lltheta, np.zeros((1, len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')

    H = np.zeros((2*(num_theta-theta_m), theta_m+num_design))
    # print(f'H:{H}')
    # print(f'H.shape: {H.shape}')

    A_t = np.vstack([-np.eye(theta_m+num_design), np.eye(theta_m+num_design)])
    # print(f'A_t:{A_t}')
    # print(f'A_t.shape: {A_t.shape}')

    theta_lb = np.array([-theta_bounds[i][0] for i in range(theta_m)] + ([-j[0] for j in design_bounds] if isinstance(design_bounds, list)
                                                                         else [])).reshape(-1, 1)
    theta_ub = np.array([theta_bounds[i][1] for i in range(theta_m)] + ([j[1] for j in design_bounds] if isinstance(design_bounds, list)
                                                                        else [])).reshape(-1, 1)

    b_t = np.vstack([theta_lb, theta_ub])
    # print(f'b_t:{b_t}')
    # print(f'b_t.shape: {b_t.shape}')

    return A, b, c, H, A_t, b_t, F

In [5]:
def get_theta_bounds(flex_sol, numt, tbounds, numd: int = 0, dbounds: list = None):
    theta_bound_dict = defaultdict(dict)
    prob_dict = defaultdict(dict)
    for i in range(numt):
        A, b, c, H, A_t, b_t, F = mpformulate_theta_bounds(
            flex_sol=flex_sol, num_theta=numt, num_design=numd, theta_bounds=tbounds, design_bounds=dbounds, theta_m=i)
        # print(f'A.shape:{A.shape}')
        # print(f'b.shape: {b.shape}')
        # print(f'F.shape: {F.shape}')
        if F.size != 0:
            prob = MPLP_Program(A=A, b=b, c=c, H=H, A_t=A_t, b_t=b_t, F=F)
            prob.process_constraints()
            solution = solve_mpqp(
                problem=prob, algorithm=mpqp_algorithm.combinatorial)
            prob_dict[f't{i}'] = prob
            theta_bound_dict[f't{i}'] = solution
        else:
            linsol = linprog(c=c, A_ub=A, b_ub=b)
            prob_dict[f't{i}'] = linsol
            theta_bound_dict[f't{i}'] = [linsol.x[1], linsol.x[0]]
            # if linsol.success:
            # print("Optimal value:", linsol.fun)
            # print("Optimal x:", linsol.x)
        print(f'Finished solving for theta{i+1}')
    probs = [p for key, p in prob_dict.items()]
    sols = [sol for key, sol in theta_bound_dict.items()]

    return probs, sols

In [11]:
y1 = 1
y2 = 0
y3 = 1
y4 = 0

alpha_1 = 0.92
alpha_2 = 0.9
alpha_3 = 0.85
alpha_4 = 0.75

# d1 = 5
# d2 = 5
# d3 = 7
# d4 = 9
# nd = 0

t_bounds = [(8, 16), (3, 11)]
d_bounds = [(4, 6), (4, 6), (6, 8), (8, 10)]
nt = len(t_bounds)
nd = len(d_bounds)

m = MPModeler()

u = m.add_var(name='u')
F1 = m.add_var(name='F1')
F2 = m.add_var(name='F2')
F3 = m.add_var(name='F3')
F4 = m.add_var(name='F4')
F5 = m.add_var(name='F5')
F6 = m.add_var(name='F6')
F7 = m.add_var(name='F7')
F8 = m.add_var(name='F8')
F9 = m.add_var(name='F9')
F10 = m.add_var(name='F10')

S = m.add_param(name='S')
D = m.add_param(name='D')

d1 = m.add_param(name='d1')
d2 = m.add_param(name='d2')
d3 = m.add_param(name='d3')
d4 = m.add_param(name='d4')

# m.add_constr(F5 - d1*y1 <= u)
# m.add_constr(F3 - F5 - d2*y2 <= u)
# m.add_constr(alpha_2*F3 - (alpha_1 - alpha_2)*F5 - d3*y3 <= u)
# m.add_constr(F1 - F3 - d4*y4 <= u)
# m.add_constr(F1 - S <= u)
# m.add_constr(D - alpha_4*F1 - (alpha_2*alpha_3 - alpha_4)*F3 + alpha_3*(alpha_1 - alpha_2)*F5 <= u)

m.add_constr(F1 == F2 + F3)
m.add_constr(F3 == F4 + F5)
m.add_constr(F10 == F8 + F9)
m.add_constr(F7 == alpha_1*F5)
m.add_constr(F6 == alpha_2*F4)
m.add_constr(F8 == alpha_3*(F6 + F7))
m.add_constr(F9 == alpha_4*F2)

m.add_constr(F5 - d1*y1 <= u)
m.add_constr(F4 - d2*y2 <= u)
m.add_constr(F6 + F7 - d3*y3 <= u)
m.add_constr(F2 - d4*y4 <= u)
m.add_constr(F1 - S <= u)
m.add_constr(D - F10 <= u)

m.add_constr(t_bounds[0][0] <= S)
m.add_constr(S <= t_bounds[0][1])
m.add_constr(t_bounds[1][0] <= D)
m.add_constr(D <= t_bounds[1][1])

m.add_constr(d_bounds[0][0] <= d1)
m.add_constr(d1 <= d_bounds[0][1])
m.add_constr(d_bounds[1][0] <= d2)
m.add_constr(d2 <= d_bounds[1][1])
m.add_constr(d_bounds[2][0] <= d3)
m.add_constr(d3 <= d_bounds[2][1])
m.add_constr(d_bounds[3][0] <= d4)
m.add_constr(d4 <= d_bounds[3][1])

m.set_objective(u)

prob = m.formulate_problem()
prob.process_constraints()

solution_flexibility = solve_mpqp(
    problem=prob, algorithm=mpqp_algorithm.combinatorial)

start_time = time.time()
# prob_list, sol_list = get_theta_bounds(flex_sol=solution_flexibility, numt=nt, numd=nd, tbounds=t_bounds)
prob_list, sol_list = get_theta_bounds(flex_sol=solution_flexibility, numt=nt, numd=nd, tbounds=t_bounds, dbounds=d_bounds)
end_time = time.time()
print(f'Elapsed time for solving mp problems: {end_time-start_time}')

def joint_pdf(theta: list):
    return (1/(2*np.pi))*np.exp(-0.5*((theta[0]-12)**2 + (theta[1]-7)**2))

Finished solving for theta1
Finished solving for theta2
Elapsed time for solving mp problems: 0.15842175483703613


In [12]:
d_vector = np.array([5, 5, 7, 9])

# REDUCE nq
nq = 7

# sf_idx_gaussian = calculate_stocflexibility(sols=sol_list, nq=nq, joint_func=joint_pdf)
sf_idx_gaussian = calculate_stocflexibility(sols=sol_list, nq=nq, joint_func=joint_pdf, d_vector=d_vector)
sf_idx_smolyak = calculate_stocflexibility_smolyak(solutions=sol_list, level=16, joint_func=joint_pdf, d_vector=d_vector)

print(f'Stochastic Flexibility Index Gaussian Legendre quadrature: {sf_idx_gaussian:.4}')
print(f'Stochastic Flexibility Index Smolyak quadrature: {sf_idx_smolyak:.4}')

Gaussian Legendre Stochastic Flexibility Elapsed time: 0.0021 s
Smolyak stochastic flexibility computed in 0.0532 seconds.
Stochastic Flexibility Index Gaussian Legendre quadrature: 0.0009735
Stochastic Flexibility Index Smolyak quadrature: 0.000969


In [13]:
len(sol_list[0])

4

In [16]:
len(sol_list[1])

1